# Dam-Break Flow with the Lax and MacCormack Schemes

A dam break is the classic test of a Saint-Venant solver because it produces both a smooth rarefaction wave and a moving shock (a bore), and it has an exact solution. You will implement the Lax and MacCormack schemes from the [lesson notes](../notes/01-governing-equations-and-explicit-schemes.md) for a rectangular, frictionless, horizontal channel and compare with the exact solution.

For a rectangular channel of constant width the conservative equations reduce (after dividing by the width) to

$$\frac{\partial\mathbf U}{\partial t}+\frac{\partial\mathbf F}{\partial x}=\mathbf 0,\qquad\mathbf U=\begin{pmatrix}h\\hu\end{pmatrix},\quad\mathbf F=\begin{pmatrix}hu\\hu^2+\tfrac12gh^2\end{pmatrix}.$$

Cells marked **TODO** in the student version are left for you to fill in.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
plt.rcParams['figure.figsize'] = (8, 6)

## Setup

A 1000 m channel with a dam at $x=500$ m. Deep water ($h_L=10$ m) on the left and shallow water ($h_R=5$ m) on the right, both initially at rest.

In [ ]:
g = 9.81
Lx, x0 = 1000.0, 500.0
hL, hR = 10.0, 5.0
t_end = 30.0

def setup(nx):
    dx = Lx / nx
    x = (np.arange(nx) + 0.5) * dx
    U = np.zeros((2, nx))
    U[0] = np.where(x < x0, hL, hR)      # depth h
    return x, dx, U                      # U[1] = h*u = 0

def flux(U):
    h, q = U
    u = q / h
    return np.array([q, q * u + 0.5 * g * h**2])

## Exact solution

For a dam break onto a wet bed, a rarefaction wave travels left and a bore travels right, separated by a region of constant state $(h_m,u_m)$. Writing $c=\sqrt{gh}$, the rarefaction connects the left state to the middle state through $u_m=2(c_L-c_m)$, and the bore conserves mass and momentum, $u_m=(h_m-h_R)\sqrt{g(h_m+h_R)/(2h_mh_R)}$. Solving these two for $h_m$ gives the full solution (Stoker, 1957).

In [ ]:
def exact_solution(x, t):
    cL = np.sqrt(g * hL)
    # middle-state depth from matching the rarefaction and the bore
    f = lambda hm: 2 * (cL - np.sqrt(g * hm)) - (hm - hR) * np.sqrt(g * (hm + hR) / (2 * hm * hR))
    hm = brentq(f, hR, hL)
    cm = np.sqrt(g * hm)
    um = 2 * (cL - cm)
    s = hm * um / (hm - hR)                      # bore speed

    xi = (x - x0) / t
    h = np.empty_like(x); u = np.empty_like(x)
    left = xi <= -cL
    fan = (xi > -cL) & (xi <= um - cm)
    mid = (xi > um - cm) & (xi <= s)
    right = xi > s
    h[left], u[left] = hL, 0.0
    h[fan] = (2 * cL - xi[fan])**2 / (9 * g)
    u[fan] = 2 / 3 * (xi[fan] + cL)
    h[mid], u[mid] = hm, um
    h[right], u[right] = hR, 0.0
    return h, u, hm, um, s

h_ex, u_ex, hm, um, s = exact_solution(np.linspace(0, Lx, 1001) + 0.0, t_end)
print(f'middle state: h = {hm:.3f} m, u = {um:.3f} m/s; bore speed = {s:.3f} m/s')

## The schemes

The time step comes from the CFL condition: the fastest wave in the system moves at $|u|+c$, so $\Delta t=\text{CFL}\,\Delta x/\max(|u|+c)$ with CFL $\le1$.

**Lax:** $\mathbf U_i^{n+1}=\tfrac12(\mathbf U_{i+1}+\mathbf U_{i-1})-\dfrac{\Delta t}{2\Delta x}(\mathbf F_{i+1}-\mathbf F_{i-1})$.

**MacCormack:** predictor $\mathbf U^*_i=\mathbf U_i-\dfrac{\Delta t}{\Delta x}(\mathbf F_i-\mathbf F_{i-1})$, corrector $\mathbf U^{**}_i=\mathbf U_i-\dfrac{\Delta t}{\Delta x}(\mathbf F^*_{i+1}-\mathbf F^*_i)$, and the update $\mathbf U_i^{n+1}=\tfrac12(\mathbf U^*_i+\mathbf U^{**}_i)$.

The boundaries are not reached by waves in this problem, so each end simply copies its neighbor.

In [ ]:
def time_step(U, dx, cfl):
    h, q = U
    return ...    # TODO: cfl * dx / max(|u| + c), with u = q/h and c = sqrt(g h)

def lax_step(U, dt, dx):
    F = flux(U)
    Un = U.copy()
    Un[:, 1:-1] = 0.5 * (U[:, 2:] + U[:, :-2]) - dt / (2 * dx) * (F[:, 2:] - F[:, :-2])
    Un[:, 0], Un[:, -1] = Un[:, 1], Un[:, -2]
    return Un

def maccormack_step(U, dt, dx, eps=0.0):
    F = flux(U)
    Us = U.copy()                                     # predictor (backward difference)
    Us[:, 1:] = ...                                   # TODO
    Fs = flux(Us)
    Uss = U.copy()                                    # corrector (forward difference)
    Uss[:, :-1] = ...                                 # TODO
    Un = 0.5 * (Us + Uss)
    if eps > 0:                                       # optional artificial viscosity
        Un[:, 1:-1] += eps * (Un[:, 2:] - 2 * Un[:, 1:-1] + Un[:, :-2])
    Un[:, 0], Un[:, -1] = Un[:, 1], Un[:, -2]
    return Un

In [ ]:
def run(step, nx, cfl=0.8, **kw):
    x, dx, U = setup(nx)
    t = 0.0
    while t < t_end - 1e-12:
        dt = min(time_step(U, dx, cfl), t_end - t)
        U = step(U, dt, dx, **kw)
        t += dt
    return x, U[0], U[1] / U[0]

## Compare with the exact solution (200 cells, CFL = 0.8)

In [ ]:
nx = 200
results = {
    'Lax': run(lax_step, nx),
    'MacCormack': run(maccormack_step, nx),
}
xe = np.linspace(0, Lx, 2001)
h_ex, u_ex, *_ = exact_solution(xe, t_end)

fig, axes = plt.subplots(2, 1, sharex=True)
axes[0].plot(xe, h_ex, 'k', lw=2, label='exact')
axes[1].plot(xe, u_ex, 'k', lw=2, label='exact')
for name, (x, h, u) in results.items():
    axes[0].plot(x, h, 'o', ms=3, label=name)
    axes[1].plot(x, u, 'o', ms=3, label=name)
axes[0].set_ylabel('depth $h$ (m)'); axes[1].set_ylabel('velocity $u$ (m/s)')
axes[1].set_xlabel('$x$ (m)'); axes[0].legend()
plt.tight_layout()

Lax is stable and smooth but smears both the rarefaction and the bore. MacCormack is sharper but oscillates in the region behind the bore (the depth in the constant middle state wanders by about $\pm0.3$ m around the exact 7.27 m), as expected for a second-order scheme without a limiter. A little artificial viscosity, or a TVD correction, reduces the wiggles.

## Effect of artificial viscosity and resolution

In [ ]:
fig, ax = plt.subplots()
ax.plot(xe, h_ex, 'k', lw=2, label='exact')
for eps in [0.0, 0.05, 0.15]:
    x, h, u = run(maccormack_step, nx, eps=eps)
    ax.plot(x, h, 'o-', ms=3, lw=0.8, label=rf'MacCormack, $\epsilon={eps}$')
ax.set_xlim(500, 650); ax.set_xlabel('$x$ (m)'); ax.set_ylabel('depth $h$ (m)'); ax.legend()

Artificial viscosity damps the oscillations but also smears the fronts, so there is an optimum: in this setup $\epsilon=0.05$ lowers the error relative to none, while $\epsilon=0.15$ raises it again. Now check convergence: the error in $h$ against the exact solution as the grid is refined.

In [ ]:
def l1_error(step, nx, **kw):
    x, h, u = run(step, nx, **kw)
    h_ex, _, *_ = exact_solution(x, t_end)
    return np.mean(np.abs(h - h_ex))

nxs = np.array([100, 200, 400, 800])
fig, ax = plt.subplots(figsize=(6, 4))
for name, step, kw in [('Lax', lax_step, {}), ('MacCormack', maccormack_step, {}),
                       ('MacCormack, $\\epsilon=0.05$', maccormack_step, {'eps': 0.05})]:
    err = [l1_error(step, n, **kw) for n in nxs]
    ax.loglog(Lx / nxs, err, 'o-', label=name)
ax.set_xlabel(r'$\Delta x$ (m)'); ax.set_ylabel('mean absolute error in $h$ (m)'); ax.legend()

Both schemes converge more slowly than their formal order because the bore is a discontinuity (see the [PDE numerics notes](../../05-pdes/notes/04-upwind-lax-and-implicit-schemes.md)): here the error falls roughly as $\Delta x^{0.7}$ for Lax and about $\Delta x^{1}$ for MacCormack, and MacCormack is several times more accurate at every resolution.

## Exercises

1. Raise the CFL number through 1.0, 1.2 and 1.5. For which values does each scheme blow up? Instabilities can take many steps to become visible, so also try a longer run. Relate your answer to the $|u|+c$ wave speed.
2. Change the initial depths to $h_L=10$ m and $h_R=0.04$ m (nearly dry bed). What happens to the shock strength, and why is a nearly dry bed hard for these schemes?
3. Add a friction term $-gh\,S_f$ to the momentum equation, using Manning's equation. Treat it explicitly. How does it change the bore?
4. Implement the two-step Richtmyer (Lax–Wendroff) scheme and compare its accuracy and oscillations with MacCormack.